### Criação de um LLM do 0

#### Parte de estudo curso de pós graduação 
Autor: Getúlio Rodrigues Silva

In [1]:
#Importações de biblioteca
import re
import random
random.seed(10)
import math
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from random import *

In [2]:
# Leitura dos dados de texto (poucas frases para exemplo)
dados_texto = open('texto.txt', 'r',encoding='utf-8').read()
type(dados_texto)

str

In [3]:
print(dados_texto)

'Olá, como vai? Eu sou a Camila.\n'
'Olá, Camila, meu nome é Fernando. Muito prazer.\n'
'Prazer em conhecer você também. Como você está hoje?\n'
'Ótimo. Meu time de futebol venceu a competição.\n'
'Uau, Parabéns Fernando!\n'
'Obrigado Camila.\n'
'Vamos comer uma pizza mais tarde para celebrar?\n'
'Claro. Você recomenda algum restaurante Camila?\n'
'Sim, abriu um restaurante novo e dizem que a pizza de banana é fenomenal.\n'
'Ok. Nos encontramos no restaurante às sete da noite, pode ser?\n'
'Pode sim. Nos vemos mais tarde então.'


In [4]:
#Filtrar Caracteres Especiais
sentences = re.sub("[.,!?\\-]", '', dados_texto.lower()).split('\n')
print(sentences)

["'olá como vai eu sou a camila\\n'", "'olá camila meu nome é fernando muito prazer\\n'", "'prazer em conhecer você também como você está hoje\\n'", "'ótimo meu time de futebol venceu a competição\\n'", "'uau parabéns fernando\\n'", "'obrigado camila\\n'", "'vamos comer uma pizza mais tarde para celebrar\\n'", "'claro você recomenda algum restaurante camila\\n'", "'sim abriu um restaurante novo e dizem que a pizza de banana é fenomenal\\n'", "'ok nos encontramos no restaurante às sete da noite pode ser\\n'", "'pode sim nos vemos mais tarde então'"]


In [5]:
# Dividir as frases em palavras e criar uma lista de palavravas unicas
word_list = list(set(" ".join(sentences).split()))
word_list

['futebol',
 'a',
 'você',
 'e',
 'banana',
 "'uau",
 "então'",
 'que',
 'nos',
 'pode',
 'vai',
 'noite',
 "'olá",
 'encontramos',
 'sim',
 "prazer\\n'",
 "'vamos",
 'pizza',
 'mais',
 "'pode",
 'sou',
 "'sim",
 "celebrar\\n'",
 "fenomenal\\n'",
 "ser\\n'",
 'conhecer',
 "hoje\\n'",
 "competição\\n'",
 'algum',
 'muito',
 'como',
 'uma',
 'sete',
 "'ótimo",
 'um',
 'nome',
 'dizem',
 "fernando\\n'",
 'venceu',
 'às',
 'abriu',
 "'ok",
 'parabéns',
 'fernando',
 'recomenda',
 'para',
 'é',
 'camila',
 'está',
 'de',
 "'obrigado",
 'também',
 'eu',
 'da',
 'time',
 "'prazer",
 'comer',
 "'claro",
 "camila\\n'",
 'restaurante',
 'vemos',
 'no',
 'novo',
 'tarde',
 'em',
 'meu']

In [6]:
#Inicializar um dicionario de palavras com os tokens espacais do BERT
word_dict = {'[PAD]': 0, '[CLS]': 1, '[SEP]': 2, '[MASK]': 3}

In [7]:
# Incluir as palavras no dicionário e criar os indices
for i, w in enumerate (word_list):
    word_dict[w] = i+4
word_dict 

{'[PAD]': 0,
 '[CLS]': 1,
 '[SEP]': 2,
 '[MASK]': 3,
 'futebol': 4,
 'a': 5,
 'você': 6,
 'e': 7,
 'banana': 8,
 "'uau": 9,
 "então'": 10,
 'que': 11,
 'nos': 12,
 'pode': 13,
 'vai': 14,
 'noite': 15,
 "'olá": 16,
 'encontramos': 17,
 'sim': 18,
 "prazer\\n'": 19,
 "'vamos": 20,
 'pizza': 21,
 'mais': 22,
 "'pode": 23,
 'sou': 24,
 "'sim": 25,
 "celebrar\\n'": 26,
 "fenomenal\\n'": 27,
 "ser\\n'": 28,
 'conhecer': 29,
 "hoje\\n'": 30,
 "competição\\n'": 31,
 'algum': 32,
 'muito': 33,
 'como': 34,
 'uma': 35,
 'sete': 36,
 "'ótimo": 37,
 'um': 38,
 'nome': 39,
 'dizem': 40,
 "fernando\\n'": 41,
 'venceu': 42,
 'às': 43,
 'abriu': 44,
 "'ok": 45,
 'parabéns': 46,
 'fernando': 47,
 'recomenda': 48,
 'para': 49,
 'é': 50,
 'camila': 51,
 'está': 52,
 'de': 53,
 "'obrigado": 54,
 'também': 55,
 'eu': 56,
 'da': 57,
 'time': 58,
 "'prazer": 59,
 'comer': 60,
 "'claro": 61,
 "camila\\n'": 62,
 'restaurante': 63,
 'vemos': 64,
 'no': 65,
 'novo': 66,
 'tarde': 67,
 'em': 68,
 'meu': 69}

In [8]:
# inverter a ordem: colocar os indices como chave e as palavras como valor no dicionario
number_dict = {i:w for i, w in enumerate(word_dict)}
number_dict

{0: '[PAD]',
 1: '[CLS]',
 2: '[SEP]',
 3: '[MASK]',
 4: 'futebol',
 5: 'a',
 6: 'você',
 7: 'e',
 8: 'banana',
 9: "'uau",
 10: "então'",
 11: 'que',
 12: 'nos',
 13: 'pode',
 14: 'vai',
 15: 'noite',
 16: "'olá",
 17: 'encontramos',
 18: 'sim',
 19: "prazer\\n'",
 20: "'vamos",
 21: 'pizza',
 22: 'mais',
 23: "'pode",
 24: 'sou',
 25: "'sim",
 26: "celebrar\\n'",
 27: "fenomenal\\n'",
 28: "ser\\n'",
 29: 'conhecer',
 30: "hoje\\n'",
 31: "competição\\n'",
 32: 'algum',
 33: 'muito',
 34: 'como',
 35: 'uma',
 36: 'sete',
 37: "'ótimo",
 38: 'um',
 39: 'nome',
 40: 'dizem',
 41: "fernando\\n'",
 42: 'venceu',
 43: 'às',
 44: 'abriu',
 45: "'ok",
 46: 'parabéns',
 47: 'fernando',
 48: 'recomenda',
 49: 'para',
 50: 'é',
 51: 'camila',
 52: 'está',
 53: 'de',
 54: "'obrigado",
 55: 'também',
 56: 'eu',
 57: 'da',
 58: 'time',
 59: "'prazer",
 60: 'comer',
 61: "'claro",
 62: "camila\\n'",
 63: 'restaurante',
 64: 'vemos',
 65: 'no',
 66: 'novo',
 67: 'tarde',
 68: 'em',
 69: 'meu'}

In [9]:
# Tamanho do vocabulário
vocab_size = len(word_dict)
vocab_size

70

In [10]:
#Criar uma list de tokens fazendo loop pelas sentenças.
token_list = list()
for sentence in sentences:
    arr = [word_dict[s] for s in sentence.split()]
    token_list.append(arr)

In [11]:
#primeira frase
dados_texto[0:29]

"'Olá, como vai? Eu sou a Cami"

In [12]:
#Pimeira frase em codigo de token
token_list

[[16, 34, 14, 56, 24, 5, 62],
 [16, 51, 69, 39, 50, 47, 33, 19],
 [59, 68, 29, 6, 55, 34, 6, 52, 30],
 [37, 69, 58, 53, 4, 42, 5, 31],
 [9, 46, 41],
 [54, 62],
 [20, 60, 35, 21, 22, 67, 49, 26],
 [61, 6, 48, 32, 63, 62],
 [25, 44, 38, 63, 66, 7, 40, 11, 5, 21, 53, 8, 50, 27],
 [45, 12, 17, 65, 63, 43, 36, 57, 15, 13, 28],
 [23, 18, 12, 64, 22, 67, 10]]

In [13]:
#Definição de Hiperparametros
batch_size = 6
n_segments = 2
dropout = 0.2

maxlen = 100
max_pred = 7

n_layer = 6

n_heads = 12

# Tamanho da embedding 
d_model = 768

#Tamanho da dimensao feedforward: 4*d_model
d_ff = d_model *4

#Dimensao e K(=Q)V
d_k = d_v = 64

#Epochs
NUM_EPOCHS = 50

In [14]:
#Definir uma função para criar os lotes de dados

def make_batch():
    batch = []
    
    #Inicializa contadores para exemplos positivos e negativos
    positive = negative = 0
    
    # Continua até que a metade do lote seja de exemplos positivos e a outra de exemplos negatvos
    while positive != batch_size/2 or negative != batch_size/2:
        
        #Escolhe índices aleatorios para duas sentenças
        tokens_a_index, tokens_b_index = randrange(len(sentences)), randrange(len(sentences))
        
        #Recupera os tokens correspondentes aos índices
        tokens_a, tokens_b = token_list[tokens_a_index], token_list[tokens_b_index]
        
        #Prepara os ids de entrada adicionano tokens especiais [CLS] e [SEP]
        input_ids = [word_dict['[CLS]']] + tokens_a + [word_dict['[SEP]']] + tokens_b + [word_dict['[SEP]']]
        
        #Define os segments ds para diferenciar as duas sentenças
        segment_ids = [0]* (1 + len(tokens_a) + 1) + [1] * (len(tokens_b) + 1)
        
        #Calcula o número de previsões a serem feiitas (15% dos tokens)
        n_pred = min(max_pred, max(1, int(round(len(input_ids)* 0.15))))
        
        #Identifica as posições candidatas para mascaramento que não sejam [CLS] ou [SEP]
        cand_maked_pos = [i for i, token in enumerate(input_ids) if token != word_dict['[CLS]'] and token != word_dict['[SEP]']]
        
        #Embaralha as posições candidatas
        shuffle(cand_maked_pos)
        
        #Inicializa listas para tokens mascarados e suas posições
        masked_tokens, masked_pos = [], []
        
        #Mascara tokens até atingir o número e previsões desejado
        for pos in cand_maked_pos[:n_pred]:
            masked_pos.append(pos)
            masked_tokens. append(input_ids[pos])
            
            #Máscara 80% das vezes
            if random() < 0.8:
                input_ids[pos]= word_dict['[MASK]']
                
            #Substitui por outro token 10% das vezes (20% do tempo restante)
            elif random() < 0.8:
                index = randint(0, vocab_size - 1)
                input_ids[pos] = word_dict[number_dict[index]]
        
        # Adiciona zero padding aos ids de entrrada e segment ids para atingir o comprimento máximo
        n_pad = maxlen - len(input_ids)
        input_ids.extend([0] * n_pad)
        segment_ids.extend([0] * n_pad)
            
        #Adiciona zero padding aos tokens mascarados e sua posições se necessário
        if max_pred > n_pred:
            n_pad = max_pred - n_pred
            masked_tokens.extend([0] * n_pad)
            masked_pos.extend([0] * n_pad)
            
        #Adiciona ao lote como um exemplo positivo se as sentenças forem consecutivas
        if tokens_a_index + 1 == tokens_b_index and positive < batch_size / 2:
            batch.append([input_ids, segment_ids, masked_tokens, masked_pos, True])
            positive += 1
        
        #Adiciona ao lote como um exemplo negativo se as sentenças não forem consecutivas
        elif tokens_a_index + 1 != tokens_b_index and negative < batch_size /2 :
            batch.append([input_ids, segment_ids, masked_tokens, masked_pos, False])
            negative += 1
    
    # Retorna o batch completo        
    return batch

In [ ]:
# Função para o padding
def get_attn_pad_masked(seq_q, seq_k):
    batch_size, len_q = seq_q.size()
    
    batch_size, len_k = seq_k.size()
    
    pad_attn_masked = seq_k.data.eq(0).unsqueeze(1)
    
    return pad_attn_masked.expand(batch_size, len_q, len_k)